# 06 — Product demonstration

A read-only client view of the analytical product core: monitored coverage,
one real time series, development workload versus recall, fault-family
coverage and ranked operational cases. It never fits a model and never opens
sealed holdout truth.


## 1. Choose sector and load outputs


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run3",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"

import duckdb
import plotly.express as px
import plotly.graph_objects as go

EDA_VERSION = "2.3.0"
VERSION = "2.3.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / f"{SECTOR}_eda_v2_3_run1"
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{VERSION}" / SECTOR / f"{SECTOR}_models_v2_3_run1"
CASE_ROOT = DATA_ROOT / "outputs" / "cases" / f"v{VERSION}" / SECTOR / f"{SECTOR}_cases_v2_3_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{VERSION}" / SECTOR / f"{SECTOR}_evaluation_v2_3_run1"

metric_summary = pd.read_csv(EDA_ROOT / "metric_summary.csv")
readiness = pd.read_parquet(MODEL_ROOT / "development_readiness.parquet")
comparison = pd.read_csv(MODEL_ROOT / "development_comparison.csv")
configuration = read_json(MODEL_ROOT / "selected_configuration.json")
cases = pd.read_csv(CASE_ROOT / "ranked_cases.csv")
metrics = pd.read_csv(CASE_ROOT / "evaluation_metrics.csv")
fault_types = pd.read_csv(CASE_ROOT / "fault_type_results.csv")
resolution = pd.read_csv(EVALUATION_ROOT / "statistical_resolution.csv")
clustered_recall = pd.read_csv(MODEL_ROOT / "clustered_recall_interval.csv")
case_trace = pd.read_parquet(CASE_ROOT / "case_score_trace.parquet")
case_run = read_json(CASE_ROOT / "case_run.json")
run_manifest = read_json(RUN_ROOT / "run_manifest.json")
pack_manifest = read_json(Path(run_manifest["pack_root"]) / "pack_manifest.json")

display(pd.Series({
    "sector": SECTOR,
    "development_reference_portfolio": configuration["portfolio"],
    "development_status": configuration["selection_status"],
    "development_cases": len(cases),
    "result_partition": case_run["partition"],
    "holdout_status": "used" if case_run["partition"] == "holdout" else "sealed",
    "source_id": pack_manifest["source"]["source_id"],
}, name="value").to_frame())


## 2. Data readiness and quality


In [ ]:
readiness_chart = readiness.groupby(["channel", "status"], as_index=False).observations.sum()
fig = px.bar(readiness_chart, x="channel", y="observations", color="status",
             title="Development scoring readiness")
fig.show()
display(resolution)
display(clustered_recall)

quality = metric_summary[["metric_id", "invalid_rate", "clipped_rate"]].melt(
    "metric_id", var_name="quality", value_name="fraction"
)
px.bar(quality, x="metric_id", y="fraction", color="quality", barmode="group",
       title="Calibration data quality by metric").show()


## 3. Representative calibration time series


In [ ]:
telemetry_glob = str(CORE_ROOT / "telemetry" / "part-*.parquet")
metric_id = metric_summary.sort_values("rows", ascending=False).iloc[0].metric_id
with duckdb.connect() as connection:
    representative = connection.execute("""
        SELECT CAST(entity_id AS VARCHAR) AS entity_id,
               CAST(episode_id AS VARCHAR) AS episode_id, count(*) AS rows
        FROM read_parquet(?)
        WHERE metric_id = ? AND quality_code <> 'invalid' AND value IS NOT NULL
        GROUP BY entity_id, episode_id ORDER BY rows DESC LIMIT 1
    """, [telemetry_glob, metric_id]).df().iloc[0]
    series = connection.execute("""
        SELECT event_ts, value, quality_code FROM read_parquet(?)
        WHERE metric_id = ? AND entity_id = ? AND episode_id = ?
        ORDER BY event_ts LIMIT 20000
    """, [telemetry_glob, metric_id, representative.entity_id, representative.episode_id]).df()

px.line(series, x="event_ts", y="value",
        title=f"{metric_id} — {representative.entity_id}").show()


## 4. Detection performance at operational workload


In [ ]:
fig = px.scatter(
    comparison, x="false_case_rate_ci_high", y="event_recall",
    color="portfolio", size="cases",
    hover_data=["threshold_setting", "false_case_rate",
                "case_precision", "median_delay_seconds"],
    title="Development recall versus conservative false-case workload",
)
fig.add_vline(x=configuration["budget_gate"], line_dash="dash")
fig.show()

px.bar(
    fault_types.sort_values("recall"), x="recall", y="fault_type",
    orientation="h", color="reporting_status",
    hover_data=["detected_faults", "scoreable_faults"],
    title="Selected portfolio: recall by fault family",
).show()


## 5. Ranked cases and why the top case fired


In [ ]:
shown = cases.head(20).copy()
px.bar(
    shown.sort_values("rank", ascending=False),
    x="anomaly_evidence_score", y="case_id", orientation="h",
    color="scope_type", hover_data=["case_start", "channels", "leading_features"],
    title="Highest anomaly-evidence cases",
).show()
display(shown[[
    "rank", "case_id", "case_start", "scope_type", "scope_id",
    "affected_entity_count", "anomaly_evidence_score", "channels", "leading_features",
]])

if not case_trace.empty and not shown.empty:
    top_case = shown.iloc[0].case_id
    trace = case_trace.loc[case_trace.case_id.eq(top_case)].copy()
    figure = go.Figure()
    for channel, frame in trace.groupby("channel"):
        figure.add_trace(go.Scatter(
            x=frame.event_ts, y=frame.anomaly_score, name=f"{channel} score"
        ))
        figure.add_trace(go.Scatter(
            x=frame.event_ts, y=frame.threshold, name=f"{channel} threshold",
            line={"dash": "dash"},
        ))
    figure.update_layout(title=f"{top_case}: anomaly score versus frozen threshold")
    figure.show()

    leading_feature = trace.leading_feature.mode().iloc[0]
    evidence = trace.loc[trace.leading_feature.eq(leading_feature)]
    figure = go.Figure()
    figure.add_trace(go.Scatter(
        x=evidence.event_ts, y=evidence.observed_transformed, name="observed"
    ))
    figure.add_trace(go.Scatter(
        x=evidence.event_ts, y=evidence.expected_transformed, name="frozen expected",
        line={"dash": "dash"},
    ))
    figure.update_layout(title=f"{top_case}: {leading_feature}, observed versus expected")
    figure.show()


## 6. What can be claimed


In [ ]:
claims = {
    "telecom": (
        "Telecom uses synthetic telemetry and generated ground truth. It validates "
        "pipeline mechanics and controlled fault tests, not real-world fibre realism."
    ),
    "petrobras_3w": (
        "Petrobras 3W uses public real-well recordings and curated dataset labels. "
        "It is the primary real-data validation sector, within the dataset's scope."
    ),
}
display(Markdown(f"""
- {claims[SECTOR]}
- The same canonical contract and evaluation mechanics run across both sectors.
- Development results are selection evidence, not final performance.
- Holdout truth remains sealed until the configuration is frozen.
- Anomaly evidence is statistical; operational priority requires validated impact data.
"""))
